In [101]:
import pandas as pd

df = pd.read_csv("data/raw/gdp_UttarPradesh1.csv")

print(df.shape)
print(df.head())
print(df.columns)

(13, 72)
      Year       Description     Agra  Aligarh  Allahabad  Ambedkar Nagar  \
0  1999-00  GDP (in Rs. Cr.)  4545.01  3774.32    5230.28         1276.49   
1  2000-01  GDP (in Rs. Cr.)  4372.74  3912.89    5639.85         1364.63   
2  2001-02  GDP (in Rs. Cr.)  4597.93  3719.35    5539.63         1465.87   
3  2002-03  GDP (in Rs. Cr.)  5058.60  4096.34    5621.03         1427.04   
4  2003-04  GDP (in Rs. Cr.)  5421.66  4296.58    6116.85         1605.11   

   Auraiyya  Azamgarh   Badaun   Bagpat  ...  Sant Kabeer Nagar  \
0   1386.52   2667.54  3588.23  1833.61  ...             920.60   
1   1443.05   2652.50  3244.58  1962.24  ...             893.29   
2   1113.67   2779.15  3473.48  1923.65  ...             952.19   
3   1173.62   2622.80  3326.80  2023.61  ...             943.93   
4   1221.24   2955.54  3587.55  2132.58  ...            1052.90   

   Sant Ravidas Nagar  Shahjahanpur  Shravasti  Siddharth Nagar  Sitapur  \
0             1687.79       2790.60     664.08   

In [102]:
df["Description"].value_counts()

Description
GDP (in Rs. Cr.)       7
Growth Rate % (YoY)    6
Name: count, dtype: int64

In [103]:
gdp_df = df[df["Description"] == "GDP (in Rs. Cr.)"].copy()
growth_df = df[df["Description"] == "Growth Rate % (YoY)"].copy()

print(gdp_df.shape)
print(growth_df.shape)

(7, 72)
(6, 72)


In [104]:
gdp_df.drop(columns=["Description"], inplace=True)
growth_df.drop(columns=["Description"], inplace=True)

In [105]:
def fix_year(y):
    start = int(y.split("-")[0])

    # handle 00–99 properly
    if start >= 90:
        return 1900 + start   # 1990–1999
    else:
        return 2000 + start   # 2000–2015

In [106]:
gdp_long = gdp_df.melt(id_vars=["Year"], var_name="District", value_name="GDP")

In [107]:
growth_long = growth_df.melt(id_vars=["Year"], var_name="District", value_name="GrowthRate")

In [108]:
final_df = pd.merge(gdp_long, growth_long, on=["Year", "District"])

print(final_df.head())

      Year District      GDP  GrowthRate
0  2000-01     Agra  4372.74   -3.790311
1  2001-02     Agra  4597.93    5.149860
2  2002-03     Agra  5058.60   10.019074
3  2003-04     Agra  5421.66    7.177085
4  2004-05     Agra  5915.91    9.116212


In [109]:
final_df["GDP"] = pd.to_numeric(final_df["GDP"], errors="coerce")
final_df["GrowthRate"] = pd.to_numeric(final_df["GrowthRate"], errors="coerce")

final_df.dropna(inplace=True)

In [110]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

final_df[["GDP_norm", "Growth_norm"]] = scaler.fit_transform(
    final_df[["GDP", "GrowthRate"]]
)

In [111]:
final_df["Development_Index"] = (
    0.7 * final_df["GDP_norm"] +
    0.3 * final_df["Growth_norm"]
)

In [112]:
final_df["Rank"] = final_df.groupby("Year")["Development_Index"] \
                           .rank(ascending=False)

In [113]:
final_df.sort_values(["Year", "Rank"]).head(10)

,Year,District,GDP,GrowthRate,GDP_norm,Growth_norm,Development_Index,Rank
270,2000-01,Lucknow,6700.99,19.962584,0.722028,0.538414,0.666944,1.0
162,2000-01,Ghaziabad,6394.74,8.676271,0.686403,0.413675,0.604585,2.0
240,2000-01,Kanpur: Urban,6052.55,11.989897,0.646597,0.450298,0.587708,3.0
12,2000-01,Allahabad,5639.85,7.830747,0.598589,0.404331,0.540312,4.0
306,2000-01,Meerut,5322.64,3.014984,0.561689,0.351106,0.498514,5.0
324,2000-01,Muzaffarnagar,5107.04,-0.111095,0.536609,0.316556,0.470593,6.0
354,2000-01,Saharanpur,4640.64,5.397229,0.482354,0.377435,0.450879,7.0
96,2000-01,BulandShahar,4641.58,4.092072,0.482464,0.363010,0.446628,8.0
390,2000-01,Sitapur,3652.60,25.797730,0.367419,0.602905,0.438065,9.0
228,2000-01,Kannauj,2153.57,61.726782,0.193042,1.000000,0.435129,10.0


In [114]:
final_df.shape

(420, 8)

In [115]:
final_df.head(420)

,Year,District,GDP,GrowthRate,GDP_norm,Growth_norm,Development_Index,Rank
0,2000-01,Agra,4372.74,-3.790311,0.451190,0.275892,0.398601,13.0
1,2001-02,Agra,4597.93,5.149860,0.477386,0.374701,0.446581,10.0
2,2002-03,Agra,5058.60,10.019074,0.530974,0.428516,0.500237,7.0
3,2003-04,Agra,5421.66,7.177085,0.573208,0.397106,0.520377,7.0
4,2004-05,Agra,5915.91,9.116212,0.630702,0.418538,0.567053,5.0
...,...,...,...,...,...,...,...,...
415,2001-02,Varanasi,3043.34,-4.517874,0.296546,0.267851,0.287937,25.0
416,2002-03,Varanasi,3249.38,6.770193,0.320514,0.392609,0.342142,18.0
417,2003-04,Varanasi,3549.53,9.237147,0.355429,0.419874,0.374763,17.0
418,2004-05,Varanasi,3712.37,4.587650,0.374372,0.368487,0.372606,17.0


In [116]:
import os

In [117]:
def load_gdp_file(path):

    df = pd.read_csv(path)

    # split metrics
    gdp = df[df["Description"] == "GDP (in Rs. Cr.)"].copy()
    growth = df[df["Description"] == "Growth Rate % (YoY)"].copy()

    # remove description
    gdp.drop(columns=["Description"], inplace=True)
    growth.drop(columns=["Description"], inplace=True)

    # clean year
    def fix_year(y):

        y = str(y)

        start = int(y.split("-")[0])

    # already normal year
        if start >= 1900:
         return start

    # formats like 99-00, 00-01
        if start >= 90:
         return 1900 + start

        return 2000 + start

    gdp["Year"] = gdp["Year"].apply(fix_year)
    growth["Year"] = growth["Year"].apply(fix_year)


    # convert wide → long

    gdp_long = gdp.melt(
        id_vars=["Year"],
        var_name="District",
        value_name="GDP"
    )


    growth_long = growth.melt(
        id_vars=["Year"],
        var_name="District",
        value_name="GrowthRate"
    )


    # merge

    final = pd.merge(
        gdp_long,
        growth_long,
        on=["Year","District"],
        how="outer"
    )


    return final

In [118]:
up1 = load_gdp_file(
    "data/raw/gdp_UttarPradesh1.csv"
)

up2 = load_gdp_file(
    "data/raw/gdp_UttarPradesh2.csv"
)

In [119]:
print(up1.shape)
print(up2.shape)

(490, 4)
(576, 4)


In [120]:
up = pd.concat(
    [up1, up2],
    ignore_index=True
)

In [121]:
up.shape

(1066, 4)

In [122]:
up = up.drop_duplicates(
    subset=["Year","District"],
    keep="last"
)

In [123]:
up["State"] = "Uttar Pradesh"

In [124]:
up.to_csv(
    "data/cleaned/up_gdp_clean.csv",
    index=False
)

In [125]:
up.head()

,Year,District,GDP,GrowthRate,State
0,1999,Agra,4545.01,NaN,Uttar Pradesh
1,1999,Aligarh,3774.32,NaN,Uttar Pradesh
2,1999,Allahabad,5230.28,NaN,Uttar Pradesh
3,1999,Ambedkar Nagar,1276.49,NaN,Uttar Pradesh
4,1999,Auraiyya,1386.52,NaN,Uttar Pradesh
